In [ ]:
# =========================
# CELL A: SET PATHS
# =========================

# ---- NI TTL folder ----
ni_folder = r"C:\Users\Shermanlab\Desktop\2026-03-05_09-27-03\Record Node 106\experiment2\recording1\events\NI-DAQmx-105.PXIe-6341\TTL"

# ---- Phy (Kilosort output) ----
phy_path = r"C:\Users\Shermanlab\Desktop\2026-03-05_09-27-03\Record Node 101\experiment2\recording1\continuous\Neuropix-PXI-100.ProbeA\kilosort4"

print("NI folder:")
print(ni_folder)

print("\nPhy path:")
print(phy_path)

In [ ]:
# =========================
# CELL 1: MODULATION SCREEN
# =========================

import numpy as np

# ---- PARAMETERS ----
pre_time = 5.0    # seconds before TTL
post_time = 10.0  # seconds after TTL

baseline_window = (-5.0, -1.0)
event_window = (0.0, 2.0)

bin_size = 0.050  # 50 ms bins

MI_THRESHOLD = 0.2     # adjust sensitivity
MIN_SPIKES = 50        # skip low-firing units

# ---- LOAD TTL TIMES (in seconds) ----
# You already have this somewhere
ttl_times = ttl_times_sec   # <-- make sure this exists

selected_units = []
modulation_info = {}

print("\n🔍 Screening units for modulation...\n")

for unit in unit_ids:

    spike_train = sorting_good.get_unit_spike_train(unit)
    spike_times = spike_train / sf

    if len(spike_times) < MIN_SPIKES:
        continue

    # ---- align spikes to TTLs ----
    aligned_spikes = []

    for t in ttl_times:
        rel_spikes = spike_times - t
        mask = (rel_spikes >= -pre_time) & (rel_spikes <= post_time)
        aligned_spikes.append(rel_spikes[mask])

    # flatten
    aligned_spikes = np.concatenate(aligned_spikes)

    # ---- compute rates ----
    baseline_mask = (aligned_spikes >= baseline_window[0]) & (aligned_spikes < baseline_window[1])
    event_mask = (aligned_spikes >= event_window[0]) & (aligned_spikes < event_window[1])

    baseline_rate = np.sum(baseline_mask) / (len(ttl_times) * (baseline_window[1] - baseline_window[0]))
    event_rate = np.sum(event_mask) / (len(ttl_times) * (event_window[1] - event_window[0]))

    # ---- modulation index ----
    if (baseline_rate + event_rate) == 0:
        MI = 0
    else:
        MI = (event_rate - baseline_rate) / (baseline_rate + event_rate)

    if abs(MI) >= MI_THRESHOLD:
        selected_units.append(unit)
        modulation_info[unit] = {
            "MI": MI,
            "baseline_rate": baseline_rate,
            "event_rate": event_rate
        }

print(f"✅ Units passing threshold: {len(selected_units)} / {len(unit_ids)}")

In [ ]:
# =========================
# CELL 2: PLOT MODULATED UNITS
# =========================

import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d

MAX_PLOTS = 30  # safety limit

units_to_plot = selected_units[:MAX_PLOTS]

print(f"\n📊 Plotting {len(units_to_plot)} units...\n")

for unit in units_to_plot:

    spike_train = sorting_good.get_unit_spike_train(unit)
    spike_times = spike_train / sf

    aligned = []

    for t in ttl_times:
        rel = spike_times - t
        mask = (rel >= -pre_time) & (rel <= post_time)
        aligned.append(rel[mask])

    # =========================
    # RASTER
    # =========================
    fig, (ax_raster, ax_peth) = plt.subplots(2, 1, figsize=(7, 6), sharex=True)

    for i, trial in enumerate(aligned):
        ax_raster.scatter(trial, np.ones_like(trial)*i, s=2)

    ax_raster.axvline(0, linestyle='--')
    ax_raster.set_ylabel("Trial")
    ax_raster.set_title(f"Unit {unit} | MI={modulation_info[unit]['MI']:.2f}")

    # =========================
    # PETH (Hz)
    # =========================
    bins = np.arange(-pre_time, post_time, bin_size)

    counts = np.zeros(len(bins)-1)

    for trial in aligned:
        h, _ = np.histogram(trial, bins=bins)
        counts += h

    rate = counts / (len(aligned) * bin_size)

    # smoothing
    rate_smooth = gaussian_filter1d(rate, sigma=2)

    centers = (bins[:-1] + bins[1:]) / 2

    ax_peth.plot(centers, rate_smooth)
    ax_peth.axvline(0, linestyle='--')

    ax_peth.set_xlabel("Time (s)")
    ax_peth.set_ylabel("Firing rate (Hz)")

    plt.tight_layout()
    plt.show()

In [ ]:
# =========================
# CELL 3: Z-SCORED HEATMAPS
# =========================

import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d

# ---- PARAMETERS ----
bin_size = 0.050
sigma = 2  # smoothing

units = selected_units  # from previous cell

if len(units) == 0:
    print("⚠️ No units selected. Run Cell 1 first.")
else:

    print(f"\n🔥 Building heatmaps for {len(units)} units...\n")

    all_z = []
    unit_MI = []

    for unit in units:

        spike_train = sorting_good.get_unit_spike_train(unit)
        spike_times = spike_train / sf

        # align to TTLs
        aligned = []
        for t in ttl_times:
            rel = spike_times - t
            mask = (rel >= -pre_time) & (rel <= post_time)
            aligned.append(rel[mask])

        # PETH
        bins = np.arange(-pre_time, post_time, bin_size)
        counts = np.zeros(len(bins)-1)

        for trial in aligned:
            h, _ = np.histogram(trial, bins=bins)
            counts += h

        rate = counts / (len(aligned) * bin_size)
        rate_smooth = gaussian_filter1d(rate, sigma=sigma)

        # ---- Z-SCORE ----
        centers = (bins[:-1] + bins[1:]) / 2

        baseline_mask = (centers >= -5.0) & (centers < -1.0)

        baseline_mean = np.mean(rate_smooth[baseline_mask])
        baseline_std = np.std(rate_smooth[baseline_mask])

        if baseline_std == 0:
            z = np.zeros_like(rate_smooth)
        else:
            z = (rate_smooth - baseline_mean) / baseline_std

        all_z.append(z)

        # store MI for sorting
        unit_MI.append(abs(modulation_info[unit]["MI"]))

    all_z = np.array(all_z)
    unit_MI = np.array(unit_MI)

    # =========================
    # PLOT 1: UNSORTED
    # =========================
    plt.figure(figsize=(8, 6))

    plt.imshow(all_z, aspect='auto', extent=[-pre_time, post_time, 0, len(units)])

    plt.axvline(0, linestyle='--')

    plt.title("Z-scored PETH (unsorted)")
    plt.xlabel("Time (s)")
    plt.ylabel("Unit")

    plt.colorbar(label="Z-score")

    plt.tight_layout()
    plt.show()

    # =========================
    # PLOT 2: SORTED BY MODULATION
    # =========================
    sort_idx = np.argsort(unit_MI)[::-1]

    all_z_sorted = all_z[sort_idx]

    plt.figure(figsize=(8, 6))

    plt.imshow(all_z_sorted, aspect='auto', extent=[-pre_time, post_time, 0, len(units)])

    plt.axvline(0, linestyle='--')

    plt.title("Z-scored PETH (sorted by modulation strength)")
    plt.xlabel("Time (s)")
    plt.ylabel("Unit (sorted)")

    plt.colorbar(label="Z-score")

    plt.tight_layout()
    plt.show()